In [ ]:
run_id = ""  
pipeline_name = ""

In [1]:
# Cell 1: Configuration — Spark connector + Catalog API

# Input tables (star schema)
DIM_PRODUCTS = "dim_products"
DIM_CATEGORIES = "dim_categories"
FACT_REVIEWS = "fact_reviews"
FACT_PRICE_CHANGES = "fact_price_changes"

# Cosmos DB configuration
COSMOS_ENDPOINT = "https://b660da96-5115-487e-9f42-e0bcc92108d4.zb6.sql.cosmos.fabric.microsoft.com:443/"
COSMOS_DATABASE = "ProductCatalog"

# Spark connector auth config (reused for reads and writes)
cosmos_base_config = {
    "spark.cosmos.accountendpoint": COSMOS_ENDPOINT,
    "spark.cosmos.database": COSMOS_DATABASE,
    "spark.cosmos.accountDataResolverServiceName": "com.azure.cosmos.spark.fabric.FabricAccountDataResolver",
    "spark.cosmos.auth.type": "AccessToken",
    "spark.cosmos.useGatewayMode": "true",
    "spark.cosmos.auth.aad.audience": "https://cosmos.azure.com/",
}

# Configure the Catalog API (used to create containers programmatically)
spark.conf.set("spark.sql.catalog.cosmosCatalog", "com.azure.cosmos.spark.CosmosCatalog")
spark.conf.set("spark.sql.catalog.cosmosCatalog.spark.cosmos.accountEndpoint", COSMOS_ENDPOINT)
spark.conf.set("spark.sql.catalog.cosmosCatalog.spark.cosmos.auth.type", "AccessToken")
spark.conf.set("spark.sql.catalog.cosmosCatalog.spark.cosmos.useGatewayMode", "true")
spark.conf.set("spark.sql.catalog.cosmosCatalog.spark.cosmos.accountDataResolverServiceName",
               "com.azure.cosmos.spark.fabric.FabricAccountDataResolver")

# Create target containers if they don't exist (idempotent, via Catalog API)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS cosmosCatalog.{COSMOS_DATABASE}.`product-insights`
    USING cosmos.oltp
    TBLPROPERTIES(partitionKeyPath = '/categoryName', autoScaleMaxThroughput = '1000')
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS cosmosCatalog.{COSMOS_DATABASE}.`category-dashboard`
    USING cosmos.oltp
    TBLPROPERTIES(partitionKeyPath = '/categoryName', autoScaleMaxThroughput = '1000')
""")

print("✅ Containers created (or already exist): product-insights, category-dashboard")

# Initialize metastore User Data Functions
from notebookutils import mssparkutils
metastore = notebookutils.udf.getFunctions('metastore_functions')


StatementMeta(, f64406f7-d4d4-4f82-a129-68f25b29ec05, 5, Finished, Available, Finished)

✅ Containers created (or already exist): product-insights, category-dashboard


In [3]:
# Cell 2: Join dim_products with aggregated fact_reviews → Cosmos DB documents
from pyspark.sql import functions as F

dim_products = spark.read.table(DIM_PRODUCTS)
fact_reviews = spark.read.table(FACT_REVIEWS)

# Aggregate reviews per product
review_agg = (
    fact_reviews.groupBy("product_key")
    .agg(
        F.round(F.avg("stars"), 2).alias("avg_rating"),
        F.count("*").alias("review_count"),
        F.sum(F.when(F.col("stars") == 5, 1).otherwise(0)).alias("five_star_count"),
        F.sum(F.when(F.col("stars") == 4, 1).otherwise(0)).alias("four_star_count"),
        F.sum(F.when(F.col("stars") == 3, 1).otherwise(0)).alias("three_star_count"),
        F.sum(F.when(F.col("stars") == 2, 1).otherwise(0)).alias("two_star_count"),
        F.sum(F.when(F.col("stars") == 1, 1).otherwise(0)).alias("one_star_count"),
        F.sum(F.when(F.col("sentiment_bucket") == "positive", 1).otherwise(0)).alias("positive_reviews"),
        F.sum(F.when(F.col("sentiment_bucket") == "negative", 1).otherwise(0)).alias("negative_reviews"),
    )
)

# Shape the data for Cosmos DB consumption
# Each document is a self-contained product insight card
product_insights = (
    dim_products.join(review_agg, dim_products.product_key == review_agg.product_key, "left")
    .select(
        dim_products.product_key.alias("id"),
        dim_products.category_name.alias("categoryName"),
        F.lit("product-insight").alias("docType"),
        dim_products.product_name.alias("productName"),
        dim_products.current_price.alias("currentPrice"),
        dim_products.inventory,
        dim_products.inventory_status.alias("inventoryStatus"),
        dim_products.price_change_pct.alias("priceChangePct"),
        dim_products.price_change_direction.alias("priceTrend"),
        F.coalesce(review_agg.avg_rating, F.lit(0.0)).alias("avgRating"),
        F.coalesce(review_agg.review_count, F.lit(0)).alias("reviewCount"),
        F.when(F.coalesce(review_agg.review_count, F.lit(0)) < 3, True).otherwise(False).alias("lowConfidence"),
        F.struct(
            F.coalesce("five_star_count", F.lit(0)).alias("fiveStar"),
            F.coalesce("four_star_count", F.lit(0)).alias("fourStar"),
            F.coalesce("three_star_count", F.lit(0)).alias("threeStar"),
            F.coalesce("two_star_count", F.lit(0)).alias("twoStar"),
            F.coalesce("one_star_count", F.lit(0)).alias("oneStar"),
        ).alias("ratingDistribution"),
        F.current_timestamp().cast("string").alias("computedAt"),
        F.lit(run_id).alias("pipelineRunId"),
    )
)

# Write to Cosmos DB via Spark connector (bulk upsert)
write_config = {**cosmos_base_config, "spark.cosmos.container": "product-insights",
                "spark.cosmos.write.strategy": "ItemOverwrite"}

product_insights.write.format("cosmos.oltp") \
    .options(**write_config) \
    .mode("APPEND") \
    .save()

print(f"✅ Product insights written to Cosmos DB: {product_insights.count()} documents")


StatementMeta(, f64406f7-d4d4-4f82-a129-68f25b29ec05, 7, Finished, Available, Finished)

✅ Product insights written to Cosmos DB: 180 documents


In [4]:
# Cell 3: Join dim_categories with aggregated facts → Cosmos DB documents
dim_categories = spark.read.table(DIM_CATEGORIES)

# Aggregate reviews by category
category_review_agg = (
    fact_reviews.groupBy("category_key")
    .agg(
        F.round(F.avg("stars"), 2).alias("avg_rating"),
        F.count("*").alias("total_reviews"),
    )
)

# Aggregate product metrics by category
category_product_agg = (
    dim_products
    .join(dim_categories.select("category_name", "category_key"), "category_name")
    .groupBy("category_key")
    .agg(
        F.count("*").alias("product_count"),
        F.round(F.avg("current_price"), 2).alias("avg_price"),
        F.round(F.min("current_price"), 2).alias("min_price"),
        F.round(F.max("current_price"), 2).alias("max_price"),
        F.sum("inventory").alias("total_inventory"),
    )
)

# Shape for Cosmos DB: each document is a category KPI card
category_dashboard = (
    dim_categories
    .join(category_product_agg, "category_key", "left")
    .join(category_review_agg, "category_key", "left")
    .select(
        F.col("category_key").alias("id"),
        F.col("category_name").alias("categoryName"),
        F.lit("category-kpi").alias("docType"),
        F.col("parent_category").alias("parentCategory"),
        F.col("sub_category").alias("subCategory"),
        F.coalesce("product_count", F.lit(0)).alias("productCount"),
        F.struct(
            F.col("avg_price").alias("average"),
            F.col("min_price").alias("minimum"),
            F.col("max_price").alias("maximum"),
        ).alias("pricing"),
        F.col("total_inventory").alias("totalInventory"),
        F.struct(
            F.col("avg_rating").alias("avgRating"),
            F.col("total_reviews").alias("totalReviews"),
        ).alias("customerSentiment"),
        F.current_timestamp().cast("string").alias("computedAt"),
        F.lit(run_id).alias("pipelineRunId"),
    )
)

# Write to Cosmos DB via Spark connector (bulk upsert)
write_config_cat = {**cosmos_base_config, "spark.cosmos.container": "category-dashboard",
                    "spark.cosmos.write.strategy": "ItemOverwrite"}

category_dashboard.write.format("cosmos.oltp") \
    .options(**write_config_cat) \
    .mode("APPEND") \
    .save()

print(f"✅ Category dashboard written to Cosmos DB: {category_dashboard.count()} documents")


StatementMeta(, f64406f7-d4d4-4f82-a129-68f25b29ec05, 8, Finished, Available, Finished)

✅ Category dashboard written to Cosmos DB: 18 documents


In [5]:
# Cell 4: Verify writes and log metadata
try:
    # Read back from Cosmos DB via Spark connector to verify
    read_pi = {**cosmos_base_config, "spark.cosmos.container": "product-insights",
               "spark.cosmos.read.inferSchema.enabled": "true"}
    read_cd = {**cosmos_base_config, "spark.cosmos.container": "category-dashboard",
               "spark.cosmos.read.inferSchema.enabled": "true"}

    product_count = spark.read.format("cosmos.oltp").options(**read_pi).load().count()
    category_count = spark.read.format("cosmos.oltp").options(**read_cd).load().count()

    print(f"Verification - product-insights: {product_count} documents")
    print(f"Verification - category-dashboard: {category_count} documents")

    import json

    summary = {
        "run_id": run_id,
        "product_insights_written": product_count,
        "category_dashboard_written": category_count,
        "status": "success",
    }
    print(json.dumps(summary, indent=2))

    # Log data quality for reverse ETL output
    metastore.log_data_quality(
        datasetId="product-insights", runId=run_id,
        totalInput=spark.read.table("dim_products").count(),
        totalOutput=product_count,
        rejected=0,
        rulesResults=[{"ruleId": "RE001", "name": "AllProductsWritten", "passed": True}])

    # Log transform lineage for reverse ETL
    metastore.log_transform_lineage(
        datasetId="product-insights", runId=run_id,
        sourceDatasets=["dim_products", "fact_reviews"],
        transforms=["aggregate_reviews", "join_products", "shape_cosmos_doc"],
        columnsAdded=["avgRating", "reviewCount", "lowConfidence", "ratingDistribution"])
    metastore.log_transform_lineage(
        datasetId="category-dashboard", runId=run_id,
        sourceDatasets=["dim_categories", "fact_reviews", "fact_price_changes"],
        transforms=["aggregate_by_category", "compute_kpis", "shape_cosmos_doc"],
        columnsAdded=["avgRating", "totalReviews", "avgPrice", "productCount"])

except Exception as e:
    raise

StatementMeta(, f64406f7-d4d4-4f82-a129-68f25b29ec05, 9, Finished, Available, Finished)

Verification - product-insights: 180 documents
Verification - category-dashboard: 18 documents
{
  "run_id": "",
  "product_insights_written": 180,
  "category_dashboard_written": 18,
  "status": "success"
}
